# Baseline Evaluation — Nemotron-3-Nano-30B (No LoRA)

Runs the base model on `train.csv`, scores predictions against ground truth, and visualises where the model succeeds and fails.

## 0 · Config

In [ ]:
import os

# ── environment ────────────────────────────────────────────────────────────────
ON_KAGGLE = os.path.exists('/kaggle')

# Model: Kaggle exposes it as a mounted dataset; locally point to a HF repo/cache
MODEL_PATH = (
    '/kaggle/input/nvidia-nemotron-nano-30b-model/transformers/default/1'
    if ON_KAGGLE
    else 'nvidia/Nemotron-3-30B'          # HuggingFace model ID (adjust if needed)
)

# How many training examples to run inference on (use None for all)
EVAL_SAMPLE = 200

# Inference params — match competition exactly
MAX_TOKENS       = 7680
TEMPERATURE      = 0.0
TOP_P            = 1.0
MAX_MODEL_LEN    = 8192
GPU_MEM_UTIL     = 0.85
MAX_NUM_SEQS     = 64

RANDOM_SEED = 42

## 1 · Install & imports

In [ ]:
# Uncomment once per environment
# !pip install -q kagglehub vllm pandas matplotlib seaborn

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import re
import random
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.metric import extract_answer, evaluate

sns.set_theme(style='whitegrid', palette='muted')
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 2 · Load dataset

In [ ]:
if ON_KAGGLE:
    DATA_DIR = pathlib.Path('/kaggle/input/nvidia-nemotron-model-reasoning-challenge')
else:
    import kagglehub
    DATA_DIR = pathlib.Path(
        kagglehub.competition_download('nvidia-nemotron-model-reasoning-challenge')
    )

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train: {len(train):,} rows | Test: {len(test):,} rows')
train.head(3)

## 3 · Exploratory Data Analysis

In [ ]:
# Basic stats
train['prompt_len']  = train['prompt'].str.len()
train['answer_len']  = train['answer'].astype(str).str.len()
train['answer_str']  = train['answer'].astype(str)
train['is_numeric']  = pd.to_numeric(train['answer'], errors='coerce').notna()

print(train[['prompt_len', 'answer_len']].describe().round(1))
print(f"\nNumeric answers : {train['is_numeric'].sum():,} / {len(train):,}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Prompt length distribution
axes[0].hist(train['prompt_len'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Prompt length (chars)')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Count')

# Answer length distribution
axes[1].hist(train['answer_len'], bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('Answer length (chars)')
axes[1].set_xlabel('Characters')

# Numeric vs string answers
val_counts = train['is_numeric'].value_counts()
axes[2].bar(['Numeric', 'String'], [val_counts.get(True, 0), val_counts.get(False, 0)],
            color=['#4CAF50', '#F44336'], edgecolor='white')
axes[2].set_title('Answer type')
axes[2].set_ylabel('Count')

plt.suptitle('Dataset Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Top-20 most frequent answers
top_answers = train['answer_str'].value_counts().head(20)

plt.figure(figsize=(10, 5))
top_answers.plot(kind='bar', color='slateblue', edgecolor='white')
plt.title('Top-20 Most Frequent Ground-Truth Answers')
plt.xlabel('Answer')
plt.ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Show a few example puzzles
for _, row in train.sample(3, random_state=RANDOM_SEED).iterrows():
    print('─' * 70)
    print(textwrap.fill(row['prompt'], width=70))
    print(f"\n→ ANSWER: {row['answer']}")
    print()

## 4 · Load model (vLLM)

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model                 = MODEL_PATH,
    max_model_len         = MAX_MODEL_LEN,
    gpu_memory_utilization= GPU_MEM_UTIL,
    max_num_seqs          = MAX_NUM_SEQS,
    trust_remote_code     = True,
)

sampling_params = SamplingParams(
    temperature = TEMPERATURE,
    top_p       = TOP_P,
    max_tokens  = MAX_TOKENS,
)

print('Model loaded.')

## 5 · Inference on training subset

In [ ]:
SYSTEM_PROMPT = (
    'You are an expert at logical and mathematical reasoning. '
    'Think step by step and place your final answer inside \\boxed{}.'
)

def build_prompt(puzzle: str) -> str:
    return (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f'<|user|>\n{puzzle}<|end|>\n'
        f'<|assistant|>\n'
    )

# Sample subset for evaluation
eval_df = train.sample(EVAL_SAMPLE, random_state=RANDOM_SEED) if EVAL_SAMPLE else train
eval_df = eval_df.reset_index(drop=True)

prompts = [build_prompt(p) for p in eval_df['prompt']]
print(f'Running inference on {len(prompts)} examples...')

In [ ]:
outputs = llm.generate(prompts, sampling_params)

raw_responses   = [o.outputs[0].text for o in outputs]
extracted_preds = [extract_answer(r) for r in raw_responses]

eval_df['raw_response'] = raw_responses
eval_df['prediction']   = extracted_preds

print('Inference complete.')

## 6 · Score & analyse

In [ ]:
from src.metric import score_prediction

results = evaluate(extracted_preds, eval_df['answer'].astype(str).tolist())

eval_df['correct'] = results['per_sample']
eval_df['extracted_answer'] = extracted_preds

print(f"Accuracy : {results['accuracy']:.2%}")
print(f"Correct  : {results['correct']} / {results['total']}")

## 7 · Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 7a — Overall accuracy
acc = results['accuracy']
axes[0].bar(['Correct', 'Wrong'], [acc, 1 - acc], color=['#4CAF50', '#F44336'], edgecolor='white')
axes[0].set_ylim(0, 1)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].set_title(f'Overall Accuracy: {acc:.1%}')

# 7b — Accuracy by answer type
type_acc = eval_df.groupby('is_numeric')['correct'].mean().rename({True: 'Numeric', False: 'String'})
type_acc.plot(kind='bar', ax=axes[1], color=['#2196F3', '#FF9800'], edgecolor='white', rot=0)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].set_ylim(0, 1)
axes[1].set_title('Accuracy by Answer Type')
axes[1].set_xlabel('')

# 7c — Accuracy by prompt length quartile
eval_df['prompt_quartile'] = pd.qcut(eval_df['prompt_len'], q=4, labels=['Q1 (short)', 'Q2', 'Q3', 'Q4 (long)'])
quartile_acc = eval_df.groupby('prompt_quartile', observed=True)['correct'].mean()
quartile_acc.plot(kind='bar', ax=axes[2], color='mediumpurple', edgecolor='white', rot=30)
axes[2].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[2].set_ylim(0, 1)
axes[2].set_title('Accuracy by Prompt Length')
axes[2].set_xlabel('Prompt length quartile')

plt.suptitle('Baseline Model Performance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 7d — Extraction failure rate
n_failed = eval_df['prediction'].isna().sum()
n_wrong_content = (~eval_df['correct'] & eval_df['prediction'].notna()).sum()
n_correct = eval_df['correct'].sum()

labels  = ['Correct', 'Wrong answer', 'No answer extracted']
sizes   = [n_correct, n_wrong_content, n_failed]
colors  = ['#4CAF50', '#F44336', '#9E9E9E']

plt.figure(figsize=(6, 6))
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140)
plt.title('Prediction Breakdown')
plt.tight_layout()
plt.show()

In [ ]:
# 7e — Response length distribution: correct vs wrong
eval_df['response_len'] = eval_df['raw_response'].str.len()

plt.figure(figsize=(9, 4))
for label, color in [('Correct', '#4CAF50'), ('Wrong', '#F44336')]:
    mask = eval_df['correct'] if label == 'Correct' else ~eval_df['correct']
    plt.hist(eval_df.loc[mask, 'response_len'], bins=40, alpha=0.6, label=label, color=color)
plt.xlabel('Response length (chars)')
plt.ylabel('Count')
plt.title('Response Length: Correct vs Wrong')
plt.legend()
plt.tight_layout()
plt.show()

## 8 · Qualitative inspection

In [ ]:
def show_examples(df, correct: bool, n: int = 3):
    subset = df[df['correct'] == correct].sample(min(n, len(df[df['correct'] == correct])),
                                                  random_state=RANDOM_SEED)
    tag = '✓ CORRECT' if correct else '✗ WRONG'
    for _, row in subset.iterrows():
        print('=' * 70)
        print(f'[{tag}]')
        print(f'PROMPT:\n{textwrap.fill(row["prompt"][:400], 70)}{"..." if len(row["prompt"]) > 400 else ""}')
        print(f'\nGROUND TRUTH : {row["answer"]}')
        print(f'EXTRACTED    : {row["prediction"]}')
        print(f'RESPONSE (first 300 chars):')
        print(textwrap.fill(row['raw_response'][:300], 70))
        print()

print('─── CORRECT examples ───────────────────────────────────────────────')
show_examples(eval_df, correct=True)

print('\n─── WRONG examples ─────────────────────────────────────────────────')
show_examples(eval_df, correct=False)

## 9 · Save results

In [ ]:
out_path = pathlib.Path('../data/baseline_predictions.csv')
out_path.parent.mkdir(exist_ok=True)
eval_df[['id', 'prompt', 'answer', 'prediction', 'correct', 'response_len']].to_csv(out_path, index=False)
print(f'Saved to {out_path}')